In [1]:
!pip install -U transformers accelerate huggingface_hub
!pip install -q streamlit faiss-gpu-cu11 ultralytics
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 78.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled

In [2]:
import torch
import gc
from transformers import CLIPProcessor, CLIPModel, Blip2Processor, Blip2ForConditionalGeneration
from huggingface_hub import snapshot_download

print("[INFO] Starting Offline Caching Protocol...")

# 1. Download BLIP-2 directly to the Hugging Face cache
print("Downloading BLIP-2...")
snapshot_download(repo_id="Salesforce/blip2-opt-2.7b")
print("[SUCCESS] BLIP-2 safely cached on hard drive!")

# 2. Download Pre-Trained CLIP directly to the Hugging Face cache
print("Downloading Pre-Trained CLIP...")
snapshot_download(repo_id="openai/clip-vit-base-patch32")
print("[SUCCESS] Pre-Trained CLIP safely cached on hard drive!")

# 3. Clear any residual memory
gc.collect()
torch.cuda.empty_cache()

print("\nDOWNLOADS COMPLETE! Your models are permanently cached for this session.")

[INFO] Starting Offline Caching Protocol...


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

[SUCCESS] BLIP-2 safely cached on hard drive!


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

[SUCCESS] Pre-Trained CLIP safely cached on hard drive!

DOWNLOADS COMPLETE! Your models are permanently cached for this session.


In [5]:
%%writefile app.py
import streamlit as st
import os, json, torch, faiss, re
import numpy as np
from PIL import Image
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel, Blip2Processor, Blip2ForConditionalGeneration
from ultralytics import YOLO

# --- CONFIGURATION & PATHS ---
st.set_page_config(page_title="Visual Search Engine", layout="wide")

# Kaggle-specific paths
BASE_DIR = "/kaggle/input/datasets/dveers/vr-final-ds"
META_PATH = "/kaggle/input/datasets/dveers/vr-final-ds/gallery_metadata.json"  
YOLO_PATH = "/kaggle/input/models/dveers/yolov11-vr-final/pytorch/default/3/runs/detect/train/weights/best.pt"
SEED_DIR = "/kaggle/input/models/dveers/finetuned-clip-vr-final/pytorch/default/2/finetuned_clip_full_543"
INDEX_DIR = "/kaggle/input/datasets/dveers/vr-final-ds/all_faiss_indices" 

# Dual GPU routing (Kaggle has 2x 16GB T4 GPUs)
DEVICE_YOLO = "cuda:0"
DEVICE_BLIP = "cuda:1"

# --- HELPER FUNCTIONS ---
def calculate_metrics(retrieved_item_ids, ground_truth_id, k_values=[5, 10, 15]):
    """Calculates retrieval metrics for a single Ground Truth ID."""
    results = {'Recall': {}, 'NDCG': {}, 'mAP': {}}
    for k in k_values:
        top_k = retrieved_item_ids[:k]
        relevance = [1 if item == ground_truth_id else 0 for item in top_k]
        
        results['Recall'][k] = 1 if sum(relevance) > 0 else 0
        dcg = sum([rel / np.log2(idx + 2) for idx, rel in enumerate(relevance)])
        results['NDCG'][k] = dcg / 1.0  
        precisions = [hits / (i + 1) for i, (rel, hits) in enumerate(zip(relevance, np.cumsum(relevance))) if rel == 1]
        results['mAP'][k] = sum(precisions) / min(len(top_k), sum(relevance) + 1e-6) if precisions else 0
    return results

def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union (IoU) for two bounding boxes [x1, y1, x2, y2]."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

# --- CACHED MODEL LOADERS ---
@st.cache_data
def load_metadata():
    with open(META_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

@st.cache_resource
def load_yolo():
    return YOLO(YOLO_PATH)

@st.cache_resource
def load_blip():
    print(f"⏳ [SERVER] Starting BLIP-2 download/load to GPU 1...")
    processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b", 
        torch_dtype=torch.float16, 
        device_map={"": 1}
    )
    print(f"[SERVER] BLIP-2 successfully loaded!")
    return processor, model

@st.cache_resource
def load_clip(mode="pretrained"):
    print(f"[SERVER] Loading {mode.upper()} CLIP to GPU 0...")
    path = "openai/clip-vit-base-patch32" if mode == "pretrained" else SEED_DIR
    processor = CLIPProcessor.from_pretrained(path)
    model = CLIPModel.from_pretrained(path).to(DEVICE_YOLO)
    print(f"[SERVER] {mode.upper()} CLIP loaded!")
    return processor, model

@st.cache_resource
def load_faiss(index_name):
    return faiss.read_index(os.path.join(INDEX_DIR, index_name))

# --- PRE-LOAD EVERYTHING AT STARTUP ---
with st.spinner("Warming up GPUs and loading models..."):
    metadata = load_metadata()
    yolo_model = load_yolo()
    blip_proc, blip_mod = load_blip()
    clip_pre_proc, clip_pre_mod = load_clip("pretrained")
    clip_ft_proc, clip_ft_mod = load_clip("finetuned")

def get_yolo_box(img, yolo):
    img_rgb = img.convert('RGB')
    results = yolo(img_rgb, conf=0.25, verbose=False, device=0)
    if len(results[0].boxes) > 0:
        return results[0].boxes.data[results[0].boxes.data[:, 4].argmax()][:4].tolist()
    return [0, 0, img.width, img.height]

# --- SESSION STATE INITIALIZATION ---
if "step" not in st.session_state:
    st.session_state.step = "upload"
if "query_image" not in st.session_state:
    st.session_state.query_image = None
if "crop_box" not in st.session_state:
    st.session_state.crop_box = None
if "final_crop" not in st.session_state:
    st.session_state.final_crop = None

# --- UI FRONTEND ---
st.title("🛒 Fashion Visual Search Engine")

st.sidebar.header("Search Settings")
condition = st.sidebar.selectbox("Select Condition", ["Part A: Vision-Only", "Part B: Frozen CLIP + Text", "Part C: Fine-Tuned CLIP + Text"])

alpha = 1.0
if condition != "Part A: Vision-Only":
    alpha = st.sidebar.radio("Select Fusion Alpha", [0.5, 0.8])

st.sidebar.markdown("---")
st.sidebar.header("Evaluation (Optional)")

# 1. Retrieval Ground Truth
ground_truth_id = st.sidebar.text_input("Ground Truth Item ID", help="Enter a single item_id (e.g., id_00000001) for Recall/NDCG/mAP metrics.")

# 2. Bounding Box Ground Truth
gt_bbox_file = st.sidebar.file_uploader("Upload GT Bounding Box (.txt)", type=["txt"], help="Upload a text file containing (x1, y1, x2, y2) to evaluate YOLO IoU.")

gt_bbox = None
if gt_bbox_file is not None:
    content = gt_bbox_file.read().decode("utf-8")
    # Extract the first 4 numbers (handles integers and floats)
    numbers = re.findall(r'-?\d+\.?\d*', content)
    if len(numbers) >= 4:
        gt_bbox = [float(numbers[0]), float(numbers[1]), float(numbers[2]), float(numbers[3])]
        st.sidebar.success(f"Loaded GT Box: {gt_bbox}")
    else:
        st.sidebar.error("Could not parse 4 coordinates from the text file.")

# --- STEP 1: UPLOAD & YOLO DETECTION ---
uploaded_file = st.file_uploader("Upload a Query Image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    if st.session_state.query_image != uploaded_file.name:
        st.session_state.query_image = uploaded_file.name
        img = Image.open(uploaded_file)
        st.session_state.crop_box = get_yolo_box(img, yolo_model)
        st.session_state.step = "confirm_crop"

    img = Image.open(uploaded_file)
    
    # --- STEP 2: CONFIRM / RE-CROP ---
    if st.session_state.step == "confirm_crop":
        st.subheader("Step 1: Product Localization")
        st.write("YOLO detected the main product. Please confirm the crop or adjust manually.")
        
        # --- IoU CALCULATION DISPLAY ---
        if gt_bbox is not None and st.session_state.crop_box is not None:
            iou = calculate_iou(st.session_state.crop_box, gt_bbox)
            st.info(f"🎯 **YOLO Detection IoU Score:** `{iou:.4f}`")
        
        col1, col2 = st.columns(2)
        with col1:
            st.image(img, caption="Original Image", width="stretch")
        
        with col2:
            box = st.session_state.crop_box
            st.markdown("**Manual Re-Crop Adjustments**")
            x1 = st.slider("Left (x1)", 0, img.width, int(box[0]))
            y1 = st.slider("Top (y1)", 0, img.height, int(box[1]))
            x2 = st.slider("Right (x2)", x1, img.width, int(box[2]))
            y2 = st.slider("Bottom (y2)", y1, img.height, int(box[3]))
            
            preview_crop = img.crop((x1, y1, x2, y2))
            st.image(preview_crop, caption="Cropped Preview", width=200)
            
            if st.button("✅ Confirm Crop & Search", width="stretch"):
                st.session_state.final_crop = preview_crop
                st.session_state.step = "search"
                st.rerun()

    # --- STEP 3: EXECUTE PIPELINE ---
    if st.session_state.step == "search":
        st.subheader("Step 2: Retrieval & Semantic Re-Ranking")
        st.image(st.session_state.final_crop, caption="Confirmed Query Image", width=200)
        
        with st.status("Running Search Pipeline...", expanded=True) as status:
            st.write("Routing to pre-loaded models...")
            
            if condition == "Part A: Vision-Only":
                active_clip_proc, active_clip_mod = clip_pre_proc, clip_pre_mod
                index = load_faiss("gallery_index_pretrained.faiss")
                use_blip = False
            elif condition == "Part B: Frozen CLIP + Text":
                active_clip_proc, active_clip_mod = clip_pre_proc, clip_pre_mod
                alpha_str = "05" if alpha == 0.5 else "08"
                index = load_faiss(f"gallery_index_pretrained_alpha{alpha_str}.faiss")
                use_blip = True
            else:
                active_clip_proc, active_clip_mod = clip_ft_proc, clip_ft_mod
                alpha_str = "05" if alpha == 0.5 else "08"
                index = load_faiss(f"gallery_index_finetuned_543_alpha{alpha_str}.faiss")
                use_blip = True

            st.write("Extracting Features (CLIP) & Searching FAISS...")
            with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                inputs = active_clip_proc(images=st.session_state.final_crop, return_tensors="pt").to(DEVICE_YOLO)
                vision_out = active_clip_mod.vision_model(pixel_values=inputs.pixel_values)
                query_embed = active_clip_mod.visual_projection(vision_out.pooler_output)
                query_embed = query_embed / query_embed.norm(p=2, dim=-1, keepdim=True)
                
                _, indices = index.search(query_embed.to(torch.float32).cpu().numpy(), 15)
                candidates = [metadata[i] for i in indices[0]]
                
            if use_blip:
                st.write("Computing Semantic Re-Ranking (BLIP-2 on GPU 1)...")
                with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    pixel_values = blip_proc(images=st.session_state.final_crop, return_tensors="pt").pixel_values.to(DEVICE_BLIP, torch.float16)
                    pixel_values = pixel_values.expand(len(candidates), -1, -1, -1)
                    
                    captions = [c['generated_caption'] for c in candidates]
                    text_inputs = blip_proc.tokenizer(captions, return_tensors="pt", padding=True, truncation=True).to(DEVICE_BLIP)
                    
                    input_ids, attention_mask = text_inputs.input_ids, text_inputs.attention_mask
                    labels = input_ids.clone()
                    labels[labels == blip_proc.tokenizer.pad_token_id] = -100
                    
                    outputs = blip_mod(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    
                    logits = outputs.logits
                    shift_logits = logits[..., :-1, :].contiguous()
                    shift_labels = labels[..., 1:].contiguous()
                    loss_matrix = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction='none')
                    loss_matrix = loss_matrix.view(shift_labels.size())
                    mask = (shift_labels != -100)
                    losses = ((loss_matrix * mask).sum(dim=1) / mask.sum(dim=1)).tolist()
                    
                    reranked = [(losses[i], candidates[i]) for i in range(len(candidates))]
                    reranked.sort(key=lambda x: x[0])
                    final_candidates = [item[1] for item in reranked]
            else:
                final_candidates = candidates
            
            status.update(label="Search Complete!", state="complete", expanded=False)

        # --- METRICS DISPLAY ---
        if ground_truth_id.strip():
            st.markdown("### 📊 Live Retrieval Metrics")
            retrieved_ids = [item['item_id'] for item in final_candidates[:15]]
            metrics = calculate_metrics(retrieved_ids, ground_truth_id.strip())
            
            m_cols = st.columns(3)
            for i, k in enumerate([5, 10, 15]):
                with m_cols[i]:
                    st.markdown(f"**Metrics @K={k}**")
                    st.write(f"Recall: `{metrics['Recall'][k]:.3f}`")
                    st.write(f"NDCG: `{metrics['NDCG'][k]:.3f}`")
                    st.write(f"mAP: `{metrics['mAP'][k]:.3f}`")
            st.markdown("---")
            
        st.subheader(f"Top Results (Condition: {condition})")
        
        cols = st.columns(5)
        for i, item in enumerate(final_candidates[:15]):
            img_path = os.path.join(BASE_DIR, "img", item['image_path'])
            try:
                res_img = Image.open(img_path)
                with cols[i % 5]:
                    # Highlight match in green if ground truth is provided
                    is_match = (ground_truth_id.strip() == item['item_id']) if ground_truth_id.strip() else False
                    border_color = "🟢 Match" if is_match else ""
                    
                    st.image(res_img, caption=f"Rank {i+1}\nID: {item['item_id']} {border_color}", width="stretch")
                    with st.expander("Show Metadata"):
                        st.write(item['generated_caption'])
            except FileNotFoundError:
                with cols[i % 5]:
                    st.error("Image not found.")
        
        if st.button("Start New Search", width="stretch"):
            st.session_state.step = "upload"
            st.rerun()

Overwriting app.py


In [6]:
# 1. Install the Ngrok Python wrapper
!pip install -q pyngrok

import subprocess
import time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

# 2. Authenticate your tunnel
user_secrets = UserSecretsClient()
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

# Kill any existing ngrok processes just to be safe
ngrok.kill()

# 3. Start Streamlit in the background
print("Starting Streamlit in the background...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.headless=true"])
time.sleep(4) # Wait a few seconds for Streamlit to boot

# 4. Open the Tunnel
print("Opening secure Ngrok tunnel...")
public_url = ngrok.connect(8501)
print(f"\n[SUCCESS] YOUR APP IS LIVE AT: {public_url.public_url}")

Starting Streamlit in the background...




2026-05-15 17:20:32.305 Uvicorn server started on 0.0.0.0:8502



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.19.2.2:8502
  External URL: http://35.188.213.99:8502

Opening secure Ngrok tunnel...

[SUCCESS] YOUR APP IS LIVE AT: https://shrapnel-facility-sabotage.ngrok-free.dev
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 3960.63it/s]


⏳ [SERVER] Starting BLIP-2 download/load to GPU 1...


Loading weights: 100%|██████████| 1247/1247 [01:12<00:00, 17.25it/s] 


[SERVER] BLIP-2 successfully loaded!
[SERVER] Loading PRETRAINED CLIP to GPU 0...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 15959.05it/s]


[SERVER] PRETRAINED CLIP loaded!
[SERVER] Loading FINETUNED CLIP to GPU 0...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 528.58it/s]
[transformers] Accessing `__path__` from `.models.aria.image_processing_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.aria.image_processing_pil_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.auto.image_processing_auto`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.beit.image_processing_beit`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.beit.image_processing_pil_beit`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future vers